<a href="https://colab.research.google.com/github/notmakgg-ux/GPT-build-by-me/blob/main/TrainingMyGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Building my own GPT from scratch.
 * Today I am learning and building my own GPT from scratch. This will help me
learn the base needed for LLM.
* The flow of the GPT is as follows :-

1.   Tokenization
2.   Embedding
3. Positional encoding
4. Attention
5. Transformer Block
6. GPT model
7. Training
8. Evaluation



## 1. Tokenization

In [9]:
from dataclasses import dataclass
import tiktoken

In [10]:
@dataclass
class TokenizerConfig:
    name: str = "gpt2"
    vocab_size: int = 50257

In [11]:
class SimpleTokenizer:
  def __init__(self, config=None):
    self.config=config or TokenizerConfig()
    self.enc= tiktoken.get_encoding(self.config.name)
    self.eos_token="<|endoftext\>"
    self.eos_token_id=self.enc.encode(
        self.eos_token,  allowed_special={self.eos_token}
    )[0]
  def encode(self, text):
    return self.enc.encode(text , allowed_special={self.eos_token})

  def decode(self,ids):
    return self.enc.decode(ids)

  @property
  def vocab_size(self):
    return self.config.vocab_size

<>:5: SyntaxWarning: invalid escape sequence '\>'
<>:5: SyntaxWarning: invalid escape sequence '\>'
/tmp/ipykernel_6988/3421483214.py:5: SyntaxWarning: invalid escape sequence '\>'
  self.eos_token="<|endoftext\>"


In [12]:
tokenizer = SimpleTokenizer()
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"EOS token: {tokenizer.eos_token}")
print(f"EOS token ID: {tokenizer.eos_token_id}")
print()

text="I am a Gangstar with a gun."
print(f"Original text: {text}")
encoded = tokenizer.encode(text)
print(f"Encoded text: {encoded}")
decoded = tokenizer.decode(encoded)
print(f"Decoded text: {decoded}")
print(f"Roundtrip: {text == decoded}")


Vocab size: 50257
EOS token: <|endoftext\>
EOS token ID: 27

Original text: I am a Gangstar with a gun.
Encoded text: [40, 716, 257, 19228, 7364, 351, 257, 2485, 13]
Decoded text: I am a Gangstar with a gun.
Roundtrip: True


In [13]:
rare = tokenizer.encode("antidisestablishmentarianism")
pieces = [tokenizer.decode([t]) for t in rare]
print(f"Pieces: {pieces}")
print(f"Decoded: '{tokenizer.decode(rare)}'")
print()

text = tokenizer.encode("Hello world!")
print(f"Decoded: {tokenizer.decode(text)}")
print()
emoji = tokenizer.encode("Hello \U0001f60a world")
print(f"Emoji test: {tokenizer.decode(emoji)}")

Pieces: ['ant', 'idis', 'establishment', 'arian', 'ism']
Decoded: 'antidisestablishmentarianism'

Decoded: Hello world!

Emoji test: Hello 😊 world


## 2. Embedding

In [14]:
import torch
import torch.nn as nn
import math

In [15]:
class Embedding(nn.Module):
  def __init__(self, vocab_size, d_model):
    super().__init__()
    self.embed=nn.Embedding(vocab_size,d_model)

  def forward(self,x):
    return self.embed(x)


In [18]:
vocab_size = 1000
d_model = 768

embedding = Embedding(vocab_size, d_model)
print(f"Embedding table shape: [{vocab_size}, {d_model}]")
print(f"Total parameters: {sum(p.numel() for p in embedding.parameters()):,}")
print()

token_ids = torch.tensor([[123, 45, 678, 464, 890]])
print(f"Input token IDs: {token_ids}")
print(f"Input shape: {token_ids.shape}")
print()

output = embedding(token_ids)
print(f"Output shape: {output.shape}")
print(f"Each token is now a {d_model}-dimensional vector")
print(f"First token vector preview: {output[0, 0, :5].tolist()} ...")

Embedding table shape: [1000, 768]
Total parameters: 768,000

Input token IDs: tensor([[123,  45, 678, 464, 890]])
Input shape: torch.Size([1, 5])

Output shape: torch.Size([1, 5, 768])
Each token is now a 768-dimensional vector
First token vector preview: [0.389218270778656, -0.25960472226142883, 1.7515723705291748, -1.1836599111557007, -0.49302583932876587] ...
